In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
df = pd.read_csv("l2_data_btcusdt.csv")

# Ensure all numeric columns are numeric for stable feature calculations.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [25]:
df.columns

# Columns
# b_p_0 is the price of the 0th best bid
# b_q_0 is the quantity of the 0th best bid
# a_p_0 is the price of the 0th best ask
# a_q_0 is the quantity of the 0th best ask
# mid_price is the mid price of the order book
# timestamp is the timestamp of the data point

Index(['timestamp', 'mid_price', 'b_p_0', 'b_p_1', 'b_p_2', 'b_p_3', 'b_p_4',
       'b_q_0', 'b_q_1', 'b_q_2', 'b_q_3', 'b_q_4', 'a_p_0', 'a_p_1', 'a_p_2',
       'a_p_3', 'a_p_4', 'a_q_0', 'a_q_1', 'a_q_2', 'a_q_3', 'a_q_4'],
      dtype='str')

In [26]:
df.head()

,timestamp,mid_price,b_p_0,b_p_1,b_p_2,b_p_3,b_p_4,b_q_0,b_q_1,b_q_2,...,a_p_0,a_p_1,a_p_2,a_p_3,a_p_4,a_q_0,a_q_1,a_q_2,a_q_3,a_q_4
0,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.74157,0.00042,0.00120,...,75488.88,75488.89,75489.31,75489.96,75490.38,4.18309,0.54611,0.00007,0.01072,0.00300
1,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54529,0.00056,0.00120,...,75488.88,75488.89,75489.31,75490.08,75490.38,4.04595,0.41582,0.00007,0.00007,0.00300
2,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54371,0.00049,0.00120,...,75488.88,75488.89,75489.13,75489.31,75490.38,4.04437,0.41582,0.00014,0.00007,0.00300
3,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54373,0.00049,0.00120,...,75488.88,75488.89,75489.31,75490.38,75490.83,4.05852,0.41582,0.00007,0.00300,0.00007
4,1.776566e+09,75444.04,75444.03,75444.02,75443.72,75442.98,75442.29,1.16658,0.00014,0.00007,...,75444.04,75444.05,75444.06,75444.07,75444.08,1.50205,0.00014,0.00014,0.43755,0.00007


In [27]:
# Convert timestamp to datetime, index by time, and enforce time order.
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

In [28]:
df.head()

,timestamp,mid_price,b_p_0,b_p_1,b_p_2,b_p_3,b_p_4,b_q_0,b_q_1,b_q_2,...,a_p_0,a_p_1,a_p_2,a_p_3,a_p_4,a_q_0,a_q_1,a_q_2,a_q_3,a_q_4
datetime,,,,,,,,,,,,,,,,,,,,,
2026-04-19 02:28:14.974487066,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.74157,0.00042,0.00120,...,75488.88,75488.89,75489.31,75489.96,75490.38,4.18309,0.54611,0.00007,0.01072,0.00300
2026-04-19 02:28:15.949206114,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54529,0.00056,0.00120,...,75488.88,75488.89,75489.31,75490.08,75490.38,4.04595,0.41582,0.00007,0.00007,0.00300
2026-04-19 02:28:16.948793888,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54371,0.00049,0.00120,...,75488.88,75488.89,75489.13,75489.31,75490.38,4.04437,0.41582,0.00014,0.00007,0.00300
2026-04-19 02:28:18.019762039,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54373,0.00049,0.00120,...,75488.88,75488.89,75489.31,75490.38,75490.83,4.05852,0.41582,0.00007,0.00300,0.00007
2026-04-19 02:28:18.968162060,1.776566e+09,75444.04,75444.03,75444.02,75443.72,75442.98,75442.29,1.16658,0.00014,0.00007,...,75444.04,75444.05,75444.06,75444.07,75444.08,1.50205,0.00014,0.00014,0.43755,0.00007


In [10]:
len(df)

7572

## Feature Engineering: Market Microstructure

In this section, we transform raw Level-2 order book data into predictive signals by calculating market microstructure features.

* **Order Book Imbalance (OBI):** The normalized difference between the best bid and best ask sizes. It acts as an indicator of immediate buying or selling pressure at the top of the book.
  $$OBI = \frac{BidQty - AskQty}{BidQty + AskQty}$$

* **Spread:** The price gap between the lowest seller and the highest buyer. A widening spread often precedes volatility or indicates low liquidity. 
  $$Spread = BestAsk - BestBid$$

* **Top 5 Depth Skew:** The ratio of total buying pressure versus total selling pressure across all 5 visible levels, giving a broader view of market walls than OBI alone.

In [30]:
# Calculating Order Book Imbalance (Top Level)
obi_denom = (df['b_q_0'] + df['a_q_0']).replace(0, np.nan)
df['obi'] = (df['b_q_0'] - df['a_q_0']) / obi_denom

In [31]:
# Calculating Spread
df['spread'] = df['a_p_0'] - df['b_p_0']

In [32]:
# Get the Top 5 Depth Skew
# Summing bid and ask quantities, then normalizing with a zero-safe denominator.
bid_depth = df[['b_q_0', 'b_q_1', 'b_q_2', 'b_q_3', 'b_q_4']].sum(axis=1)
ask_depth = df[['a_q_0', 'a_q_1', 'a_q_2', 'a_q_3', 'a_q_4']].sum(axis=1)
depth_denom = (bid_depth + ask_depth).replace(0, np.nan)
df['depth_skew'] = (bid_depth - ask_depth) / depth_denom

In [33]:
# Get the Mid-Price Momentum (Rolling 10-second change)
# Shows if the price has been trending up or down before this snapshot
df['momentum_10s'] = df['mid_price'].pct_change(periods=10)

In [34]:
df.head()

,timestamp,mid_price,b_p_0,b_p_1,b_p_2,b_p_3,b_p_4,b_q_0,b_q_1,b_q_2,...,a_p_4,a_q_0,a_q_1,a_q_2,a_q_3,a_q_4,obi,spread,depth_skew,momentum_10s
datetime,,,,,,,,,,,,,,,,,,,,,
2026-04-19 02:28:14.974487066,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.74157,0.00042,0.00120,...,75490.38,4.18309,0.54611,0.00007,0.01072,0.00300,-0.698834,0.01,-0.729024,NaN
2026-04-19 02:28:15.949206114,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54529,0.00056,0.00120,...,75490.38,4.04595,0.41582,0.00007,0.00007,0.00300,-0.762465,0.01,-0.781652,NaN
2026-04-19 02:28:16.948793888,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54371,0.00049,0.00120,...,75490.38,4.04437,0.41582,0.00014,0.00007,0.00300,-0.762990,0.01,-0.782173,NaN
2026-04-19 02:28:18.019762039,1.776566e+09,75488.88,75488.87,75488.86,75488.00,75487.81,75487.74,0.54373,0.00049,0.00120,...,75490.83,4.05852,0.41582,0.00007,0.00300,0.00007,-0.763711,0.01,-0.782777,NaN
2026-04-19 02:28:18.968162060,1.776566e+09,75444.04,75444.03,75444.02,75443.72,75442.98,75442.29,1.16658,0.00014,0.00007,...,75444.08,1.50205,0.00014,0.00014,0.43755,0.00007,-0.125709,0.01,-0.248781,NaN


In [35]:
df.isna().sum()

timestamp        0
mid_price        0
b_p_0            0
b_p_1            0
b_p_2            0
b_p_3            0
b_p_4            0
b_q_0            0
b_q_1            0
b_q_2            0
b_q_3            0
b_q_4            0
a_p_0            0
a_p_1            0
a_p_2            0
a_p_3            0
a_p_4            0
a_q_0            0
a_q_1            0
a_q_2            0
a_q_3            0
a_q_4            0
obi              0
spread           0
depth_skew       0
momentum_10s    10
dtype: int64

In [36]:
# Drop any NaN values created by the rolling momentum calculation
df.dropna(subset=['momentum_10s'], inplace=True)

Why These Features? (Feature Engineering Logic)

Raw prices (like $64,000) are terrible inputs for Machine Learning models. They aren't stationary—if you train a model when BTC is at 60k, and it drops to 50k, the model breaks because it hasn't seen that absolute number before. 

Instead, we engineer features that are **relative** and **normalized**. These capture the underlying supply and demand mechanics regardless of the current price level.

1. Order Book Imbalance (OBI)
* **What it is:** The normalized difference between the best bid and best ask sizes.
    $$OBI = \frac{BidQty - AskQty}{BidQty + AskQty}$$
* **Why we use it:** It measures the immediate tug-of-war at the very front line (Level 0). It gives the model a clean $-1$ to $+1$ scale indicating whether buyers or sellers are more aggressive right now. 

2. Spread
* **What it is:** The absolute price gap between the best ask and best bid.
* **Why we use it:** It acts as a proxy for market volatility and liquidity. A tight spread usually means a highly liquid, calm market. A sudden widening of the spread tells the model that market makers are pulling liquidity, usually preceding a volatile move.

3. Top 5 Depth Skew
* **What it is:** The total buying pressure vs. total selling pressure summed across all 5 visible levels.
* **Why we use it:** OBI only looks at the absolute best price. Depth skew catches "walls" (massive resting orders) sitting a few levels deep. If a whale places a massive buy order 3 levels below the mid-price, OBI misses it, but depth skew flags it.

## Classification Problem??

Because live websocket timestamps are close to 1 second but not perfectly fixed, we use a true time horizon. For each row at time `t`, we find the first snapshot at or after `t + 30s` and compare its mid price to the current one. If future price is greater, label is 1 (UP), otherwise 0 (DOWN or FLAT).

In [39]:
PREDICTION_WINDOW_SEC = 30

# Build a true time-based horizon: first snapshot at or after t + 30s.
base = df[['mid_price']].copy().reset_index().rename(columns={'datetime': 't'})
base['t_plus_window'] = base['t'] + pd.Timedelta(seconds=PREDICTION_WINDOW_SEC)

future = base[['t', 'mid_price']].rename(columns={'t': 'future_t', 'mid_price': 'future_mid_price'})

merged = pd.merge_asof(
    base.sort_values('t_plus_window'),
    future.sort_values('future_t'),
    left_on='t_plus_window',
    right_on='future_t',
    direction='forward'
)

# Restore original order and write target inputs back to df.
merged.sort_index(inplace=True)
df['future_mid_price'] = merged['future_mid_price'].values
df['target'] = (df['future_mid_price'] > df['mid_price']).astype('Int64')

### Export transformed data to csv

## Train a Baseline Model

Use a chronological split (no shuffling) so we evaluate on future data, which better matches live trading conditions.

In [42]:
df.to_csv("l2_data_btcusdt_transformed.csv")